### Generating Ground Truth Data

To evaluate search, we need a dataset of queries where we know which document is the correct answer. This is called ground truth (or gold standard) data.

For each query in our ground truth dataset, we know which document in the knowledge base is relevant. When we run a search, we check whether the results include the correct document.

There are several ways to get ground truth data:

* Human annotators look at documents and write queries (best quality, expensive)
* Collect real user queries and label them (requires a running system)
* Generate synthetic data with an LLM (what we'll do)


We don't have a production system yet, so we'll use an LLM to generate questions. For each FAQ document, we ask the LLM to create 5 questions that this document would answer. Then we know that for each generated question, the source document is the correct answer.


#### Loading the documents

We'll use helper files.

Then load the FAQ data:

In [1]:
from ingest import load_data
documents = load_data()

We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ dataset contains documents from multiple courses. 

Generating five questions for every document would take longer and cost more.

In [2]:
documents

[{'mal_id': 52991,
  'title': 'Sousou no Frieren',
  'title_english': "Frieren: Beyond Journey's End",
  'type': 'TV',
  'source': 'Manga',
  'episodes': 28.0,
  'status': 'Finished Airing',
  'airing': False,
  'rating': 'PG-13 - Teens 13 or older',
  'score': 9.28,
  'scored_by': 823434,
  'rank': 1.0,
  'popularity': 108,
  'members': 1353494,
  'favorites': 85889,
  'synopsis': 'During their decade-long quest to defeat the Demon King, the members of the hero\'s party—Himmel himself, the priest Heiter, the dwarf warrior Eisen, and the elven mage Frieren—forge bonds through adventures and battles, creating unforgettable precious memories for most of them.\n\nHowever, the time that Frieren spends with her comrades is equivalent to merely a fraction of her life, which has lasted over a thousand years. When the party disbands after their victory, Frieren casually returns to her "usual" routine of collecting spells across the continent. Due to her different sense of time, she seemingly h

Each document already has an id field:

The ID becomes the label in our ground truth dataset. We generate questions from a document, 

#### Generating questions with structured output

We use an LLM to generate questions for each document.

This is the first time we're using structured output in the course. With structured output, we ask the LLM to return data in a specific format instead of free-form text. 

For example, instead of getting a paragraph that contains questions, we can ask for a Python object with a questions field.

This is useful when code will process the output. The model returns the same structure every time. We can access the generated questions directly instead of parsing text manually.

We want the output as a list of strings, so we define that structure with a Pydantic model:

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The instructions for the LLM:

In [37]:
data_gen_instructions = """
You emulate a person who needs recomendations to know what light novels, webtoons, or anime to watch.
Formulate 5 questions this person might ask based on a record. The result returned should contain the questions, and the questions should be complete and short.
The questions should not use the characters names or proper names inside the record. The questions should redirect a person to chose the anime, light novel, webtoon or anime described in the record.

The output should resemble how people ask for recommendations of what to watch on the internet. Don´t use Flowery language used by writers, use casual vocabulary, not too long, and try use diferent words than the one used in the record. For example: a- Any recommendations about anime or manga about protagonists that are females and mages? b- Any recommendations about manga or anime about magic elves? c- anyone can tell me the name of manga that talks about a magic elf that is very old and powerfull and that sees her conrades die due to age? d- Any recommendation about animes, light novels or mangas about magic, elves, and the typical travel to fight with demons and other monsters? e- I want to watch another isekai manga or anime or light novel.  
""".strip()

We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users won't phrase their questions the same way as the FAQ.

Call the LLM for one document:

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Prepare the document as JSON:

In [29]:
import json

user_prompt = json.dumps(documents[0])

Create the messages:

In [38]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. 

For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

Call the model:

In [39]:
response = openai_client.responses.parse(
    model="gpt-5.4-nano",
    input=messages,
    text_format=Questions
)

The parsed object is available in response.output_parsed:

In [40]:
result = response.output_parsed

print(result)

questions=['Any recommendations for fantasy anime about an elf mage who lives for a long time and has to deal with losing friends?', 'What should I watch next if I like slow-burn fantasy with lots of character growth and bittersweet memories?', 'Any light novels, webtoons, or anime similar to a post-quest story where a party breaks up and the main character keeps living forward?', 'Looking for anime or manga about humans and elves, with adventure plus a strong focus on relationships and regret', 'Can you recommend fantasy anime or manga that are more drama-focused than action, with a mage traveling and collecting spells?']


We can access the list directly:

In [42]:
print(result.questions)

['Any recommendations for fantasy anime about an elf mage who lives for a long time and has to deal with losing friends?', 'What should I watch next if I like slow-burn fantasy with lots of character growth and bittersweet memories?', 'Any light novels, webtoons, or anime similar to a post-quest story where a party breaks up and the main character keeps living forward?', 'Looking for anime or manga about humans and elves, with adventure plus a strong focus on relationships and regret', 'Can you recommend fantasy anime or manga that are more drama-focused than action, with a mage traveling and collecting spells?']


You should see 5 questions that relate to the first FAQ document.

#### Reusable utilities

We'll need this pattern again in other evaluation sections today, so we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

* llm_structured: calls the OpenAI API with structured output
* llm_structured_retry: retries structured-output calls when a request fails
* calc_price: calculates the price from token usage
* calc_total_price: calculates the total price from multiple usage objects
* map_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.

Import the structured-output helper:

In [43]:
from evaluation_utils import llm_structured

Use it on the same document:

In [49]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Any anime about an old elf mage thinking back on a past hero trip?', 'Any good fantasy show where the main character is super long-lived and learns about human feelings later on?', 'Looking for an anime with a party of adventurers, demons, and a lot of emotional flashbacks?', 'Any recommendations for a calm fantasy series with magic, travel, and a life after saving the world?', 'What anime is like a fantasy journey that starts after the big bad is already beaten?']


#### Tracking cost

The response also contains token usage:

In [45]:
usage.input_tokens, usage.output_tokens

(707, 104)

As in the agents module, we calculate the price from response.usage.

Import the price helper:

In [46]:
from evaluation_utils import calc_price

Calculate the cost of this call:

In [47]:
cost = calc_price(usage)

cost

{'input_cost': 0.0001414, 'output_cost': 0.00013, 'total_cost': 0.0002714}

Now convert these questions into ground truth records:

In [51]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": documents[0]["mal_id"]
    })

records

[{'question': 'Any anime about an old elf mage thinking back on a past hero trip?',
  'document': 52991},
 {'question': 'Any good fantasy show where the main character is super long-lived and learns about human feelings later on?',
  'document': 52991},
 {'question': 'Looking for an anime with a party of adventurers, demons, and a lot of emotional flashbacks?',
  'document': 52991},
 {'question': 'Any recommendations for a calm fantasy series with magic, travel, and a life after saving the world?',
  'document': 52991},
 {'question': 'What anime is like a fantasy journey that starts after the big bad is already beaten?',
  'document': 52991}]

Each record has two fields:

* question: the question generated by the LLM
* document: the ID of the FAQ document that should answer the question

The document field connects the generated question to the document that contains the answer. 

Later, when we evaluate search, we'll ask the search engine the generated question. 

Then we'll check if it retrieves the document with this ID.

We now know how to generate and store questions for one document. In the next lesson, we'll run this for all LLM Zoomcamp FAQ documents and save the full ground truth dataset.

### Generating Ground Truth for All Documents

In the previous part, we generated questions for one document and converted them into ground truth records.

We want to do the same thing for every document in the FAQ dataset. For each document, we generate questions and save them as ground truth records.


For this part, we'll use tqdm for progress bars and pandas for saving the final CSV.


If you don't have them installed yet, add them first:

uv add tqdm pandas

The processing function takes one document and turns it into ground truth records.

For each document, we:

* convert the document to JSON so we can send it to the LLM
* ask the LLM to return a Questions object
* create one ground truth record for each generated question

Each record contains the generated question and the ID of the document that should answer the question.

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

Import the retry helper from evaluation_utils.py:

In [52]:
from evaluation_utils import llm_structured_retry

llm_structured makes one structured-output call. 

llm_structured_retry wraps the same call in a retry loop. 

If one request fails because of a temporary API or network issue, it waits briefly and tries again.

Use it in the processing function:

In [77]:
#def generate_ground_truth(doc, i): #testing purposes
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            #"document": documents[i]["mal_id"] #testing purposes
            "document": doc['mal_id']
        })

    return results, usage

Try it for the first 5 documents.

Import tqdm and run the loop:

In [73]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for index, doc in enumerate(tqdm(documents[:2])):
    print(doc)
    #records, usage = generate_ground_truth(doc, index)
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/2 [00:00<?, ?it/s]

{'mal_id': 52991, 'title': 'Sousou no Frieren', 'title_english': "Frieren: Beyond Journey's End", 'type': 'TV', 'source': 'Manga', 'episodes': 28.0, 'status': 'Finished Airing', 'airing': False, 'rating': 'PG-13 - Teens 13 or older', 'score': 9.28, 'scored_by': 823434, 'rank': 1.0, 'popularity': 108, 'members': 1353494, 'favorites': 85889, 'synopsis': 'During their decade-long quest to defeat the Demon King, the members of the hero\'s party—Himmel himself, the priest Heiter, the dwarf warrior Eisen, and the elven mage Frieren—forge bonds through adventures and battles, creating unforgettable precious memories for most of them.\n\nHowever, the time that Frieren spends with her comrades is equivalent to merely a fraction of her life, which has lasted over a thousand years. When the party disbands after their victory, Frieren casually returns to her "usual" routine of collecting spells across the continent. Due to her different sense of time, she seemingly holds no strong feelings toward 

In [62]:
ground_truth

[{'question': 'Any anime like this about an elf mage dealing with life after beating the dark lord?',
  'document': 52991},
 {'question': 'Any recommendations for fantasy shows that focus more on feelings and memories than nonstop action?',
  'document': 52991},
 {'question': 'What anime has an old long-lived magic user thinking back on past adventures and lost friends?',
  'document': 52991},
 {'question': 'Looking for a fantasy series with a party journey, then a calm story after the big final battle?',
  'document': 52991},
 {'question': 'Any good anime or manga about magic, travel, and learning to understand humans over time?',
  'document': 52991},
 {'question': 'Any good fantasy anime with an elf mage traveling with a small party?',
  'document': 59978},
 {'question': 'What should I watch if I want a slow adventure anime with magic and some emotional stuff?',
  'document': 59978},
 {'question': 'Any anime about a long-lived mage learning more about people and feelings?',
  'docum

This works, but it runs one LLM call after another. Running it for all documents this way would take too long.

#### Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

Import ThreadPoolExecutor:

In [74]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

This submits one job per document, updates the progress bar when a job finishes, and collects the results. 

If you want a more detailed explanation of ThreadPoolExecutor and futures, ask ChatGPT to walk through this helper line by line.

Then replace the loop with the parallel version:

In [82]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents[:500], generate_ground_truth)

  0%|          | 0/500 [00:00<?, ?it/s]

*generate_ground_truth* returns two things for each document: the generated records and the token usage.

Split those into separate lists:

In [83]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

2500

With 5 questions per document, you should get roughly 5x the number of documents.

Calculate the total cost:

In [84]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12797225

We'll calculate total cost several times in this module, so the utility file has a helper for it:

In [85]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.12797225

Create a dataframe so we can look at the records as a table and save them as a CSV file.

Create the dataframe:

In [86]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

Because we generated the questions from specific documents, we know which document is correct for each question. 

We now have the ground truth we need for evaluation.

Save it for later use:

In [87]:
df_ground_truth.to_csv("./ground_truth-mal.csv", index=False)

Now we have questions with known correct documents. In the next part, we'll run search for these questions and check whether the correct documents appear in the results.

### Evaluating retrieval quality

Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.


#### Setting up search

We will create a new notebook for search evaluation. We'll use the ground truth CSV generated here and set up the search index there.

For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.
